In [5]:
import jax
import jax.numpy as jnp
from jax import random

jax.config.update("jax_enable_x64", True) # Crucial for higher-order derivatives

# 1. Define the Metric (The Statistical Prior/Topology)
def gaussian_metric(theta):
    """
    Fisher Information Matrix for a univariate Gaussian.
    theta = [mu, sigma]
    """
    mu, sigma = theta[0], theta[1]
    return jnp.array([
        [1.0 / sigma**2, 0.0],
        [0.0, 2.0 / sigma**2]
    ])

# 2. Differential Geometry Operators
def christoffel_symbols(metric_fn, theta):
    """Computes the Levi-Civita connection coefficients \Gamma^k_{ij}."""
    # Derivative of metric: dg[i, j, k] = \partial_k g_{ij}
    dg = jax.jacfwd(metric_fn)(theta)
    g = metric_fn(theta)
    g_inv = jnp.linalg.inv(g)
    
    # \Gamma^k_{ij} = 0.5 * g^{kl} (dg_{li, j} + dg_{lj, i} - dg_{ij, l})
    # Using einsum for tensor contractions
    term1 = jnp.einsum('kl,lij->kij', g_inv, dg)
    term2 = jnp.einsum('kl,lji->kij', g_inv, dg)
    term3 = jnp.einsum('kl,ijl->kij', g_inv, dg)
    
    gamma = 0.5 * (term1 + term2 - term3)
    return gamma

def riemann_tensor(metric_fn, theta):
    """Computes the Riemann curvature tensor R^l_{ijk}."""
    gamma_fn = lambda t: christoffel_symbols(metric_fn, t)
    gamma = gamma_fn(theta)
    
    # dgamma[l, i, j, m] = \partial_m \Gamma^l_{ij}
    dgamma = jax.jacfwd(gamma_fn)(theta)
    
    # R^l_{ijk} = \partial_j \Gamma^l_{ik} - \partial_k \Gamma^l_{ij} 
    #             + \Gamma^l_{jm}\Gamma^m_{ik} - \Gamma^l_{km}\Gamma^m_{ij}
    
    # \partial_j \Gamma^l_{ik} -> dgamma[l, i, k, j]
    term1 = jnp.einsum('likj->lijk', dgamma)
    
    # \partial_k \Gamma^l_{ij} -> dgamma[l, i, j, k]
    term2 = dgamma 
    
    # \Gamma^l_{jm}\Gamma^m_{ik} -> sum_m gamma[l, j, m] * gamma[m, i, k]
    term3 = jnp.einsum('ljm,mik->lijk', gamma, gamma)
    
    # \Gamma^l_{km}\Gamma^m_{ij} -> sum_m gamma[l, k, m] * gamma[m, i, j]
    term4 = jnp.einsum('lkm,mij->lijk', gamma, gamma)
    
    return term1 - term2 + term3 - term4
    
def ricci_and_scalar(metric_fn, theta):
    """Computes the Ricci tensor R_{ij} and Scalar Curvature R."""
    R_lijk = riemann_tensor(metric_fn, theta)
    g = metric_fn(theta)
    g_inv = jnp.linalg.inv(g)
    
    # Ricci tensor R_{ik} = R^l_{ilk}
    ricci_tensor = jnp.einsum('lilk->ik', R_lijk)
    
    # Scalar curvature R = g^{ik} R_{ik}
    scalar_curvature = jnp.einsum('ik,ik->', g_inv, ricci_tensor)
    
    return ricci_tensor, scalar_curvature

# 3. Model Scoring via Exact Evidence Expansion
def geometric_model_score(theta_mle, metric_fn, n_samples, log_likelihood, k_params):
    """
    Evaluates the truncated expansion: -log Z_M(n)
    """
    ricci, R = ricci_and_scalar(metric_fn, theta_mle)
    
    # 1. Classical BIC terms
    bic_penalty = (k_params / 2.0) * jnp.log(n_samples / (2 * jnp.pi))
    
    # 2. Robustness/Naturalness (Volume normalization)
    # det(J) / det(I) approximation - assuming empirical J ~ Fisher I at MLE for flat data
    robustness = 0.0 
    
    # 3. O(n^-1) Curvature Penalty (Using intrinsic R)
    # T_ij (Stress-Energy from data) is assumed isolated in the residual variance here.
    geom_penalty = R / n_samples
    
    # Total Score (Lower is better)
    score = -log_likelihood + bic_penalty + robustness + geom_penalty
    
    return score, R

def gmrf_precision(theta, L):
    """Constructs the precision matrix Q = alpha*I + beta*L."""
    alpha, beta = theta[0], theta[1]
    I = jnp.eye(L.shape[0])
    return alpha * I + beta * L

def gmrf_metric(theta, L):
    """
    Computes the 2x2 Fisher Information Metric for the GMRF.
    theta = [alpha, beta]
    """
    Q = gmrf_precision(theta, L)
    Q_inv = jnp.linalg.inv(Q)
    
    # Partial derivatives of Q
    dQ_dalpha = jnp.eye(L.shape[0])
    dQ_dbeta = L
    
    # Compute the Fisher metric components using the exact trace formula
    g11 = 0.5 * jnp.trace(Q_inv @ dQ_dalpha @ Q_inv @ dQ_dalpha)
    g12 = 0.5 * jnp.trace(Q_inv @ dQ_dalpha @ Q_inv @ dQ_dbeta)
    g22 = 0.5 * jnp.trace(Q_inv @ dQ_dbeta @ Q_inv @ dQ_dbeta)
    
    return jnp.array([
        [g11, g12],
        [g12, g22]
    ])

# --- Let's test it on a simple 1D Cycle Graph (Ring Network) ---
N = 10 # Number of nodes
# Construct a simple ring Laplacian (1D discrete continuum)
off_diag = jnp.diag(jnp.ones(N-1), 1) + jnp.diag(jnp.ones(N-1), -1)
off_diag = off_diag.at[0, N-1].set(1.0).at[N-1, 0].set(1.0) # Periodic boundary
L_ring = jnp.diag(jnp.sum(off_diag, axis=1)) - off_diag

# Define a wrapped metric function to pass to the geometry engine
metric_fn_gmrf = lambda t: gmrf_metric(t, L_ring)

# Evaluate at some arbitrary structural state (alpha=1.0, beta=0.5)
theta_gmrf = jnp.array([1.0, 0.5])

# (Assuming ricci_and_scalar is defined from the previous script)
ricci_tensor, scalar_R = ricci_and_scalar(metric_fn_gmrf, theta_gmrf)

print(f"Graph Size: {N} nodes")
print(f"GMRF Scalar Curvature (R): {scalar_R:.6f}")


# --- 1. Simulate "True" Data (The Mass) ---
def simulate_empirical_covariance(N, true_theta, L, n_samples, key):
    """Generates an empirical covariance matrix S from a true GMRF."""
    Q_true = gmrf_precision(true_theta, L)
    Cov_true = jnp.linalg.inv(Q_true)
    
    # Draw samples from N(0, Cov_true)
    samples = random.multivariate_normal(
        key, 
        mean=jnp.zeros(N), 
        cov=Cov_true, 
        shape=(n_samples,)
    )
    # Compute empirical covariance S
    S = jnp.cov(samples, rowvar=False)
    return S

# --- 2. The Stress-Energy Tensor ---
def empirical_stress_energy(theta, L, S):
    """Computes T_ij = U_i U_j from the empirical covariance S."""
    Q = gmrf_precision(theta, L)
    Q_inv = jnp.linalg.inv(Q)
    
    # Partial derivatives (from previous logic)
    dQ_dalpha = jnp.eye(L.shape[0])
    dQ_dbeta = L
    
    # The Score Vector U_i
    residual = Q_inv - S
    U_alpha = 0.5 * jnp.trace(residual @ dQ_dalpha)
    U_beta = 0.5 * jnp.trace(residual @ dQ_dbeta)
    
    U = jnp.array([U_alpha, U_beta])
    
    # T_ij is the outer product of the Score
    T_ij = jnp.outer(U, U)
    return T_ij

# --- Execution: Observing the Warping ---
key = random.PRNGKey(42)
N = 10
n_samples = 50

# 1. Generate the Laplacian (from the previous script)
off_diag = jnp.diag(jnp.ones(N-1), 1) + jnp.diag(jnp.ones(N-1), -1)
off_diag = off_diag.at[0, N-1].set(1.0).at[N-1, 0].set(1.0)
L_ring = jnp.diag(jnp.sum(off_diag, axis=1)) - off_diag

# 2. Simulate reality: S
true_theta = jnp.array([1.0, 0.5]) 
S_empirical = simulate_empirical_covariance(N, true_theta, L_ring, n_samples, key)

# 3. Initialize our model at a WRONG state (empty vacuum)
guess_theta = jnp.array([2.0, 0.1]) 

# 4. Compute the forces
T_tensor = empirical_stress_energy(guess_theta, L_ring, S_empirical)

print(f"Current Guess: alpha={guess_theta[0]}, beta={guess_theta[1]}")
print(f"Empirical Stress-Energy Tensor (T_ij):\n{T_tensor}")
# --- Execution ---
#theta_mle = jnp.array([0.0, 1.0]) # Standard Normal MLE
#ricci_tensor, scalar_R = ricci_and_scalar(gaussian_metric, theta_mle)

#print(f"Ricci Tensor:\n{ricci_tensor}")
#print(f"Scalar Curvature (R): {scalar_R:.5f}")

Graph Size: 10 nodes
GMRF Scalar Curvature (R): -0.000000
Current Guess: alpha=2.0, beta=0.1
Empirical Stress-Energy Tensor (T_ij):
[[ 0.25266452 -0.22941764]
 [-0.22941764  0.20830963]]
